# 03 — Preprocessing & Feature Selection
Split terstratifikasi & terkelompok (*anti-leakage*), imputasi median dari train, transformasi $\log_{10}$, one-hot encoding, scaling, serta seleksi fitur berbasis VIF dan *drop-column CV*.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from house_price.data.cleaning import load_or_build_clean_data
from house_price.features.preprocessing import (
    add_wilayah_model,
    run_preprocessing_pipeline,
    save_preprocessed_artifacts,
)
from house_price.features.selection import (
    compute_vif,
    evaluate_drop_column_cv,
    rmse_cv,
    verify_log_ratio_redundancy,
)
from house_price.config import RANDOM_STATE

df_clean = load_or_build_clean_data()
prep = run_preprocessing_pipeline(df_clean, random_state=RANDOM_STATE)
print(f"Train: {len(prep.train_df):,} baris | Test: {len(prep.test_df):,} baris")

Train: 1,107 baris | Test: 277 baris


## 1. Verifikasi Split & Pencegahan Data Leakage

In [2]:
display(pd.DataFrame({
    "train_%": prep.train_df["wilayah_model"].value_counts(normalize=True) * 100,
    "test_%": prep.test_df["wilayah_model"].value_counts(normalize=True) * 100,
}))

df_w = add_wilayah_model(df_clean)
tr_acak, te_acak = train_test_split(df_w, test_size=0.2, random_state=RANDOM_STATE, stratify=df_w["wilayah_model"])
bocor_acak = te_acak["deskripsi"].str.lower().isin(tr_acak["deskripsi"].str.lower()).sum()
bocor_grup = prep.test_df["deskripsi"].str.lower().isin(prep.train_df["deskripsi"].str.lower()).sum()
print(f"Baris test yang judulnya juga ada di train -> split acak: {bocor_acak} | grouped split: {bocor_grup}")

ks = stats.ks_2samp(prep.y_train, prep.y_test)
print(f"KS test log10(harga) train vs test: stat = {ks.statistic:.3f}, p-value = {ks.pvalue:.3f}")
print("Median train yang dipakai:", prep.median_train.to_dict())

,train_%,test_%
wilayah_model,,
Sleman,72.54,72.20
Bantul,15.63,15.88
Kota Yogyakarta,9.49,9.39
Kulon Progo/Gunung Kidul,2.35,2.53


Baris test yang judulnya juga ada di train -> split acak: 5 | grouped split: 0
KS test log10(harga) train vs test: stat = 0.058, p-value = 0.434
Median train yang dipakai: {'kamar_tidur': 3.0, 'kamar_mandi': 2.0}


## 2. Scaling & Encoding Wilayah

In [3]:
jml_area = prep.train_df["area"].value_counts()
print(f"Area unik di train: {len(jml_area)} | area dengan < 5 listing: {(jml_area < 5).sum()} ({jml_area[jml_area < 5].sum()} listing)")

display(pd.concat({
    "train sebelum": prep.X_train_unscaled[prep.kolom_scale].agg(["mean", "std"]),
    "train sesudah": prep.X_train[prep.kolom_scale].agg(["mean", "std"]),
    "test sesudah": prep.X_test[prep.kolom_scale].agg(["mean", "std"]),
}).round(3))

Area unik di train: 57 | area dengan < 5 listing: 24 (58 listing)


log_luas_tanah  log_luas_bangunan  kamar_tidur  \
train sebelum mean            2.13               2.01         3.26   
              std             0.25               0.29         1.24   
train sesudah mean            0.00               0.00        -0.00   
              std             1.00               1.00         1.00   
test sesudah  mean           -0.02              -0.05        -0.09   
              std             0.98               0.94         0.93   

                    kamar_mandi  carport  
train sebelum mean         2.33     1.21  
              std          1.24     0.97  
train sesudah mean         0.00     0.00  
              std          1.00     1.00  
test sesudah  mean        -0.06     0.04  
              std          1.01     1.30

## 3. Feature Selection (VIF & Drop-Column CV RMSE)

In [4]:
vif = compute_vif(prep.X_train_unscaled.astype(float))
display(vif.to_frame())

grup_cv = prep.train_df["deskripsi"].str.lower()
drop_imp, dasar = evaluate_drop_column_cv(
    prep.X_train,
    prep.X_train_unscaled,
    prep.y_train,
    groups=grup_cv,
    random_state=RANDOM_STATE,
)
print("RMSE CV dengan semua fitur:", {k: round(v, 4) for k, v in dasar.items()})
display(drop_imp.round(4))

r2_ratio = verify_log_ratio_redundancy(prep.X_train_unscaled)
print(f"R^2 log(LB/LT) dari log LB & log LT = {r2_ratio:.4f} -> redundan sempurna")
print("Fitur final:", prep.fitur_final)
print("Bentuk data:", prep.X_train_final.shape, prep.X_test_final.shape)

artifact_path = save_preprocessed_artifacts(prep)
print(f"Artefak preprocessing tersimpan di: {artifact_path}")

,VIF
log_luas_bangunan,3.99
kamar_mandi,3.09
kamar_tidur,3.04
log_luas_tanah,2.72
carport,1.78
carport_kosong,1.78
wilayah_Sleman,1.64
wilayah_Kota Yogyakarta,1.50
wilayah_Kulon Progo/Gunung Kidul,1.16


RMSE CV dengan semua fitur: {'Linear': 0.1631, 'RandomForest': 0.1472}


,delta_RMSE_Linear,delta_RMSE_RandomForest,delta_%error_Linear
luas bangunan,0.03,0.02,7.05
luas tanah,0.01,0.02,1.48
kamar mandi,0.00,0.00,0.81
wilayah (one-hot),0.00,0.00,0.50
kamar tidur,0.00,0.00,0.13
carport,0.00,-0.00,0.09
carport + carport_kosong,0.00,0.00,0.08
carport_kosong,0.00,-0.00,0.03


R^2 log(LB/LT) dari log LB & log LT = 1.0000 -> redundan sempurna
Fitur final: ['log_luas_bangunan', 'log_luas_tanah', 'kamar_mandi', 'wilayah_Kota Yogyakarta', 'wilayah_Kulon Progo/Gunung Kidul', 'wilayah_Sleman']
Bentuk data: (1107, 6) (277, 6)
Artefak preprocessing tersimpan di: D:\Dev\Python\Semester-5\DataScience\pPrediksi_Harga_Rumah_DS\data\processed\preprocessing_artifacts.joblib
